In [1]:
import pandas as pd

# df1 = pd.read_csv("../../data/dataset.csv", sep="\\|\\|\\|", engine="python").drop(["modern_prompt", "translation_prompt"], axis=1)
df1 = pd.read_csv("../../data/cleaned_dataset.csv", sep="|", engine="python")       # les textes
df2_init = pd.read_csv("../../data/paires_mot_lemme.csv", )                         # les mots

df2 = df2_init.rename(columns={'mot': 'old_french', 'lemme': 'modern'})



In [2]:
df1["type"] = "phrase"
df1.head()


,modern,old_french,type
0,"salut tout le monde, c'est victor. je repensai...","seigneurs et dames, je vous salue, c'est victo...",phrase
1,"salut tout le monde ! aujourd'hui, c'était un ...","saluz tout le monde ! en ce jour, fu moult bon...",phrase
2,"chère sophie, tu ne devineras jamais ce qui m'...","chère sophie, tu ne devineras ja mie ce qui m'...",phrase
3,"bonjour à tous, ici marcel dupré, artisan poti...","bien le bon jour à tous, céans marcel dupré, o...",phrase
4,"yo, c’est lila. alors voilà, l’autre jour, j’é...","salut, c'est lila. lors, l'autre jour, j'estoi...",phrase


In [3]:
df2["type"] = "mot"
df2.head()


,old_french,modern,type
0,Cil,cil,mot
1,qui,qui1,mot
2,fist,faire,mot
3,d',de,mot
4,Erec,Erec,mot


In [4]:
df = pd.concat([df1, df2], ignore_index=True)
df.shape

(33848, 3)

In [5]:
# Création des colonnes pour T5
df["input_text"] = "traduire: " + df["modern"].astype(str)
df["target_text"] = df["old_french"].astype(str)



In [6]:
# Supprime les NaN classiques
df = df.dropna()

# Supprime les lignes où "nan" apparaît comme string (dans input_text ou target_text)
mask = (df["input_text"].str.lower() != "traduire: nan") & (df["target_text"].str.lower() != "nan")
df = df[mask].reset_index(drop=True)


In [7]:
# On garde juste le nécessaire
df = df[["input_text", "target_text"]]
df = df.dropna()
print(df.sample(3))

                 input_text target_text
5521   traduire: volontiers  Volentiers
7936        traduire: cruel      cruels
28947      traduire: donner        duné


In [8]:
# Conversion HuggingFace Dataset & Split

from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)


/home/malek/BRIEFS DEV IA/14.NLP/EULA-vaaag-/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Tokenization du Dataset

from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-base")
max_input_length = 128
max_target_length = 128

def preprocess_function(batch):
    # Tokenize les inputs
    model_inputs = tokenizer(
        batch["input_text"], max_length=max_input_length, padding="max_length", truncation=True
    )
    # Tokenize les targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["target_text"], max_length=max_target_length, padding="max_length", truncation=True
        )
    # Remplace le padding des labels par -100 (ignore loss)
    labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels_ids
    return model_inputs

# Mapping correct avec batched=True
tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=["input_text", "target_text"])



You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Map:   0%|          | 0/27099 [00:00<?, ? examples/s]/home/malek/BRIEFS DEV IA/14.NLP/EULA-vaaag-/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:3959: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 3012/3012 [00:00<00:00,

In [10]:
# Chargement modèle + DataCollator

from transformers import T5ForConditionalGeneration, DataCollatorForSeq2Seq

model = T5ForConditionalGeneration.from_pretrained("t5-base")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)


2025-06-20 09:45:19.806442: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-20 09:45:19.813728: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750405519.823197    8263 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750405519.825846    8263 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750405519.833193    8263 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# Entraînement

from transformers import TrainingArguments, Trainer
import torch

# Libère mémoire GPU si besoin
if torch.cuda.is_available():
    torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir="./t5-vieux-francais",
    per_device_train_batch_size=3,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    fp16=True,  # True si GPU compatible, sinon False
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()


/tmp/ipykernel_8263/2435918769.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss
500,2.816300,2.594618
1000,2.765400,2.472072


: 

In [ ]:
# Sauvegarde du modèle et du tokenizer

trainer.save_model("./t5-vieux-francais-model")
tokenizer.save_pretrained("./t5-vieux-francais-model")


In [ ]:
# Fonction de Traduction (inférence)

import torch

def generate_translation(text, model, tokenizer, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    model.eval()

    input_text = "traduire: " + text
    inputs = tokenizer(
        input_text, return_tensors="pt", max_length=128, truncation=True, padding="max_length"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_length=128,
            num_beams=4,
            temperature=1.0,
            do_sample=False,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)



In [ ]:
# Exemple de Test

test_text = "Bonjour, comment allez-vous ?"
result = generate_translation(test_text, model, tokenizer)
print(f"Moderne: {test_text}")
print(f"Vieux français: {result}")




In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

model = T5ForConditionalGeneration.from_pretrained("../Vic/model_save")
tokenizer = T5Tokenizer.from_pretrained("../Vic/model_save")


In [ ]:
import gradio as gr

def translate_interface(text):
    return generate_translation(text, model, tokenizer)

gr.Interface(fn=translate_interface, inputs="text", outputs="text", title="Traduction en Vieux Français").launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
